<a href="https://colab.research.google.com/github/junseok-jay/AI_lab/blob/main/pipeline/model_export_IR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 시작

In [12]:
import os, torch
from torch import nn

# ====== 공통 설정 ======
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEQ = 16
BS  = 1

def show_export_ir(name: str, mod: nn.Module, example_args: tuple, example_kwargs: dict | None = None, max_lines=200):
    mod.eval()
    if example_kwargs is None:
        ep = torch.export.export(mod, example_args)
    else:
        ep = torch.export.export(mod, example_args, example_kwargs)

    g = str(ep.graph_module.graph)
    lines = g.splitlines()
    print(f"\n========== {name} (device={DEVICE}) ==========")
    print(f"Graph lines: {len(lines)} (showing last {min(max_lines, len(lines))})")
    print("\n".join(lines[-max_lines:]))

In [13]:
# ====== HF 공통 유틸 ======
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

@torch.no_grad()
def rand_ids(vocab, bs=BS, seq=SEQ):
    return torch.randint(0, vocab, (bs, seq), device=DEVICE, dtype=torch.long)

@torch.no_grad()
def ones_mask(bs=BS, seq=SEQ):
    return torch.ones((bs, seq), device=DEVICE, dtype=torch.long)

In [14]:
# ====== 1) ResNet50 ======
def export_resnet50():
    import torchvision
    m = torchvision.models.resnet50(weights=None).to(DEVICE).eval()
    x = torch.randn(BS, 3, 224, 224, device=DEVICE)
    show_export_ir("ResNet50", m, (x,), max_lines=500)

# ====== 실행 ======
export_resnet50()


========== ResNet50 (device=cuda) ==========
Graph lines: 498 (showing last 498)
graph():
    %p_conv1_weight : [num_users=1] = placeholder[target=p_conv1_weight]
    %p_bn1_weight : [num_users=1] = placeholder[target=p_bn1_weight]
    %p_bn1_bias : [num_users=1] = placeholder[target=p_bn1_bias]
    %p_layer1_0_conv1_weight : [num_users=1] = placeholder[target=p_layer1_0_conv1_weight]
    %p_layer1_0_bn1_weight : [num_users=1] = placeholder[target=p_layer1_0_bn1_weight]
    %p_layer1_0_bn1_bias : [num_users=1] = placeholder[target=p_layer1_0_bn1_bias]
    %p_layer1_0_conv2_weight : [num_users=1] = placeholder[target=p_layer1_0_conv2_weight]
    %p_layer1_0_bn2_weight : [num_users=1] = placeholder[target=p_layer1_0_bn2_weight]
    %p_layer1_0_bn2_bias : [num_users=1] = placeholder[target=p_layer1_0_bn2_bias]
    %p_layer1_0_conv3_weight : [num_users=1] = placeholder[target=p_layer1_0_conv3_weight]
    %p_layer1_0_bn3_weight : [num_users=1] = placeholder[target=p_layer1_0_bn3_weight]
  

In [15]:
# ====== 2) BERT-base-uncased (last_hidden_state Tensor만 반환) ======
class BertIRWrap(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        return out.last_hidden_state  # [B, S, H]

def export_bert_base():
    model_id = "bert-base-uncased"
    bert = AutoModel.from_pretrained(model_id).to(DEVICE).eval()
    w = BertIRWrap(bert).to(DEVICE).eval()
    input_ids = rand_ids(bert.config.vocab_size)
    attn = ones_mask()
    show_export_ir("BERT-base-uncased", w, (input_ids, attn))

# ====== 실행 ======
export_bert_base()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



========== BERT-base-uncased (device=cuda) ==========
Graph lines: 513 (showing last 200)
    %transpose_12 : [num_users=1] = call_function[target=torch.ops.aten.transpose.int](args = (%view_9, 1, 2), kwargs = {})
    %linear_19 : [num_users=1] = call_function[target=torch.ops.aten.linear.default](args = (%layer_norm_6, %p_bert_encoder_layer_3_attention_self_key_weight, %p_bert_encoder_layer_3_attention_self_key_bias), kwargs = {})
    %view_10 : [num_users=1] = call_function[target=torch.ops.aten.view.default](args = (%linear_19, [1, 16, -1, 64]), kwargs = {})
    %transpose_13 : [num_users=1] = call_function[target=torch.ops.aten.transpose.int](args = (%view_10, 1, 2), kwargs = {})
    %linear_20 : [num_users=1] = call_function[target=torch.ops.aten.linear.default](args = (%layer_norm_6, %p_bert_encoder_layer_3_attention_self_value_weight, %p_bert_encoder_layer_3_attention_self_value_bias), kwargs = {})
    %view_11 : [num_users=1] = call_function[target=torch.ops.aten.view.default]

In [17]:
# ====== 3) GPT-2 (logits Tensor만 반환, use_cache=False) ======
class CausalLMLogitsWrap(nn.Module):
    def __init__(self, clm):
        super().__init__()
        self.clm = clm
        # 캐시 끄기(그래프 흔들림 방지)
        if hasattr(self.clm, "config"):
            self.clm.config.use_cache = False
        if hasattr(self.clm, "generation_config"):
            self.clm.generation_config.use_cache = False
    def forward(self, input_ids, attention_mask):
        out = self.clm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True,
        )
        return out.logits  # [B, S, V]

def export_gpt2():
    model_id = "gpt2"
    gpt2 = AutoModelForCausalLM.from_pretrained(model_id).to(DEVICE).eval()
    w = CausalLMLogitsWrap(gpt2).to(DEVICE).eval()
    input_ids = rand_ids(gpt2.config.vocab_size)
    attn = ones_mask()
    show_export_ir("GPT-2", w, (input_ids, attn))

# ====== 실행 ======
export_gpt2()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


========== GPT-2 (device=cuda) ==========
Graph lines: 675 (showing last 200)
    %layer_norm_14 : [num_users=1] = call_function[target=torch.ops.aten.layer_norm.default](args = (%add_29, [768], %p_clm_transformer_h_7_ln_1_weight, %p_clm_transformer_h_7_ln_1_bias), kwargs = {})
    %view_79 : [num_users=1] = call_function[target=torch.ops.aten.view.default](args = (%layer_norm_14, [-1, 768]), kwargs = {})
    %addmm_28 : [num_users=1] = call_function[target=torch.ops.aten.addmm.default](args = (%p_clm_transformer_h_7_attn_c_attn_bias, %view_79, %p_clm_transformer_h_7_attn_c_attn_weight), kwargs = {})
    %view_80 : [num_users=1] = call_function[target=torch.ops.aten.view.default](args = (%addmm_28, [1, 16, 2304]), kwargs = {})
    %split_7 : [num_users=3] = call_function[target=torch.ops.aten.split.Tensor](args = (%view_80, 768, 2), kwargs = {})
    %getitem_21 : [num_users=1] = call_function[target=operator.getitem](args = (%split_7, 0), kwargs = {})
    %getitem_22 : [num_users=1] =

In [18]:
# ====== 4) Llama 3.2 3B Instruct (logits Tensor만 반환, fp16 권장) ======
def export_llama32_3b():
    model_id = "meta-llama/Llama-3.2-3B-Instruct"

    # 토큰 필요하면 환경변수 HF_TOKEN 사용
    token = os.environ.get("HF_TOKEN", None)

    llama = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map=None,
        token=token,
    ).to(DEVICE).eval()

    w = CausalLMLogitsWrap(llama).to(DEVICE).eval()
    input_ids = rand_ids(llama.config.vocab_size)
    attn = ones_mask()
    show_export_ir("Llama-3.2-3B-Instruct", w, (input_ids, attn))

# ====== 실행 ======
export_llama32_3b()

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]


========== Llama-3.2-3B-Instruct (device=cuda) ==========
Graph lines: 2045 (showing last 200)
    %add_151 : [num_users=3] = call_function[target=torch.ops.aten.add.Tensor](args = (%add_149, %linear_174), kwargs = {})
    %_assert_tensor_metadata_default_110 : [num_users=0] = call_function[target=torch.ops.aten._assert_tensor_metadata.default](args = (%add_151,), kwargs = {dtype: torch.float16, device: cuda:0, layout: torch.strided})
    %to_110 : [num_users=2] = call_function[target=torch.ops.aten.to.dtype](args = (%add_151, torch.float32), kwargs = {})
    %pow_51 : [num_users=1] = call_function[target=torch.ops.aten.pow.Tensor_Scalar](args = (%to_110, 2), kwargs = {})
    %mean_50 : [num_users=1] = call_function[target=torch.ops.aten.mean.dim](args = (%pow_51, [-1], True), kwargs = {})
    %add_152 : [num_users=1] = call_function[target=torch.ops.aten.add.Tensor](args = (%mean_50, 1e-05), kwargs = {})
    %rsqrt_50 : [num_users=1] = call_function[target=torch.ops.aten.rsqrt.defaul